# CachePilot：Colab 单 GPU Phase 1 完整验证

目标：得到 TODO 2.3 的**渐进压测、必要对照、故障验证、Phase 1 出口**四项结论。顺序执行所有单元，最后下载原始证据、`conclusions.json` 和 `report.md`。失败也保留数据，不自动勾选 TODO。

先选择「运行时 → 更改运行时类型 → GPU」，将包含本次修改和 .git 的完整仓库放入 Drive。全流程使用独立 Python 3.13.15 + 真实 TorchExecutor + 固定 Qwen revision；T4 选 float16，支持 BF16 的卡选 bfloat16。同一矩阵不切换硬件/dtype。下载和完整扫描可能需要数十分钟至数小时。

**语义边界**：Torch 当前先完成整段 generate 再交付 token，TTFT/TPOT 是 Gateway 交付指标，不能解释为逐 token GPU decode 性能；物理 prefix 命中不可观测。static/continuous 对照为 N/A。vLLM 接入检查使用另一个 `colab_acceptance.ipynb`，不混入本矩阵。

## 1. 工作目录、代码快照与日志

修改 SOURCE 后执行；代码复制到 /content，避免 Drive FUSE 安装问题。每次创建独立会话，禁止混合不同会话的重复次数。不在 notebook 填写任何凭据。后续任一步失败，先修复或直接执行最后的归档单元。

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import atexit, hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request

drive.mount("/content/drive")
SOURCE = Path("/content/drive/MyDrive/CachePilot")
SESSION = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + os.urandom(3).hex()
WORKDIR = Path("/content") / ("CachePilot-" + SESSION)
OUT = Path("/content/cachepilot-phase1") / SESSION
BACKUP = Path("/content/drive/MyDrive/CachePilot-evidence") / SESSION
assert (SOURCE / ".git").exists(), "需要完整仓库以记录 commit"
assert (SOURCE / "benchmarks/colab_phase1.py").exists(), "请先同步新增脚本"
shutil.copytree(SOURCE, WORKDIR, ignore=shutil.ignore_patterns(
    "runs", ".venv", "venv", "__pycache__", ".pytest_cache", ".ruff_cache", ".idea"))
OUT.mkdir(parents=True)
BACKUP.mkdir(parents=True)
os.chdir(WORKDIR)
os.environ["PYTHONPATH"] = str(WORKDIR)

def save(name, value):
    target = OUT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

def checkpoint():
    shutil.copytree(OUT, BACKUP, dirs_exist_ok=True)

def redact(text):
    text = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", text)
    return re.sub(r"(?i)bearer\s+\S+", "Bearer [REDACTED]", text)

def command(argv, label, timeout=3600, required=True):
    try:
        result = subprocess.run(list(map(str, argv)), cwd=WORKDIR, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    except subprocess.TimeoutExpired:
        save(label + ".status.json", {"returncode": None, "error": "timeout"})
        checkpoint()
        raise
    text = redact(result.stdout)
    (OUT / (label + ".log")).write_text(text)
    save(label + ".status.json", {"returncode": result.returncode})
    checkpoint()
    print(text[-6000:])
    if required and result.returncode:
        raise RuntimeError(label + " 失败，查看对应日志")
    return result

code_files = {}
for folder in ("cachepilot", "benchmarks", "config", "requirements", "workloads", "tests", "notebooks"):
    for path in sorted((WORKDIR / folder).rglob("*")):
        if path.is_file() and path.suffix in {".py", ".json", ".jsonl", ".txt", ".ipynb"}:
            code_files[str(path.relative_to(WORKDIR))] = hashlib.sha256(path.read_bytes()).hexdigest()
for name in ("pyproject.toml", "main.py", ".python-version"):
    code_files[name] = hashlib.sha256((WORKDIR / name).read_bytes()).hexdigest()
save("source.json", {
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "git_dirty": bool(subprocess.check_output(["git", "status", "--porcelain"], text=True)),
    "file_sha256": code_files})
checkpoint()
print("本轮证据:", BACKUP)


## 2. 独立解释器与 GPU 环境门禁

项目固定 Python **3.13.15**。通过 uv 在 /content 安装独立解释器；所有服务/实验使用 PY，不依赖 Colab 内核版本，也无需重启内核。若锁定 Python 或依赖不可下载，结论是环境阻塞；不要用 ignore-requires-python 绕过。只安装 Torch GPU 依赖，不额外安装 vLLM。

要求 Linux x86_64、恰好一张 NVIDIA GPU、至少 8 GiB 显存和 10 GiB 空闲磁盘、CUDA 可用、模型/tokenizer revision 固定。

In [ ]:
command([sys.executable, "-m", "pip", "install", "uv"], "bootstrap-uv")
UV = shutil.which("uv")
assert UV
command([UV, "python", "install", "3.13.15"], "python-install")
VENV = Path("/content") / ("cachepilot-venv-" + SESSION)
command([UV, "venv", "--python", "3.13.15", VENV], "venv")
PY = str(VENV / "bin/python")
command([UV, "pip", "install", "--python", PY, "-r", "requirements/requirements-gpu.txt",
         "pytest==8.3.5", "httpx==0.28.1"], "install")
command([PY, "-m", "benchmarks.gpu_environment", "--root", WORKDIR, "--require-torch"], "environment")
command([UV, "pip", "freeze", "--python", PY], "dependencies")
result = command([PY, "-c", "import torch; print('bfloat16' if torch.cuda.is_bf16_supported() else 'float16')"], "dtype")
DTYPE = result.stdout.strip().splitlines()[-1]
save("session.json", {"session": SESSION, "dtype": DTYPE, "python": PY, "executor": "TorchExecutor"})


## 3. Phase 0 前置出口与 API/SSE 回归

必须先通过 CPU 不变量和 API/SSE/取消/超时测试。此处包含替身后端，只证明控制路径；真实 GPU 证据来自后续步骤。

In [ ]:
command([PY, "-m", "benchmarks.phase0_exit"], "phase0", timeout=1800)
command([PY, "-m", "pytest", "-q", "tests/integration/test_gateway_api.py",
         "tests/integration/test_vllm_gateway.py", "tests/unit/test_torch_executor.py"],
        "api-regression", timeout=1800)


## 4. 服务管理与固定协议

四策略：Strict/FCFS/blind、Adaptive/FCFS/blind、Strict/WFQ/blind、Strict/WFQ/aware；分别隔离 admission、scheduler、prefix 单变量。

正式 workload：32 个请求、两 tenant、混合长度、同时到达、重复 prompt，max_tokens=32，并发16。每策略三轮，每轮**重启 → 串行 warm-up 同一 workload → 正式测量**，共12个warm-up和12个measured run；warm-up 为 Adaptive 分桶提供历史。轮转策略顺序降低时间偏差。

小样本 P99 接近最大值；需要稳健性能估计时增加 CONTEXTS/REPETITIONS 并新建会话。公平性只衡量本固定同权 tenant workload，不代表所有 noisy-neighbor 场景。

In [ ]:
BASE_URL = "http://127.0.0.1:8000"
STRATEGIES = [("strict", "fcfs", "blind"), ("adaptive", "fcfs", "blind"),
              ("strict", "wfq", "blind"), ("strict", "wfq", "aware")]
CONTEXTS = [32, 32, 128, 128, 512, 512, 128, 128] * 4
REPETITIONS, CONCURRENCY, MAX_TOKENS = 3, 16, 32
server = None
server_handle = None

def stop_server():
    global server, server_handle
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=10)
        server = None
    if server_handle is not None:
        server_handle.close()
        server_handle = None
    for path in OUT.glob("server-*.log"):
        path.write_text(redact(path.read_text(errors="replace")))
    checkpoint()

atexit.register(stop_server)

def start_server(strategy, label):
    global server, server_handle
    import socket
    stop_server()
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", 8000)) == 0:
            raise RuntimeError("8000 被其他服务占用，请停止旧 notebook 的服务")
    server_handle = (OUT / ("server-" + label + ".log")).open("w")
    server = subprocess.Popen([PY, "main.py", "--host", "127.0.0.1", "--port", "8000",
        "--dtype", DTYPE, "--admission", strategy[0], "--scheduler", strategy[1],
        "--prefix-mode", strategy[2]], cwd=WORKDIR, stdout=server_handle, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        if server.poll() is not None:
            stop_server()
            raise RuntimeError("服务启动失败，检查 server 日志")
        try:
            with urllib.request.urlopen(BASE_URL + "/readyz", timeout=2) as response:
                if json.load(response)["status"] == "ready":
                    return
        except (OSError, ValueError):
            time.sleep(1)
    stop_server()
    raise TimeoutError("加载模型超过 900 秒")

save("protocol.json", {"strategies": STRATEGIES, "contexts": CONTEXTS,
    "repetitions": REPETITIONS, "concurrency": CONCURRENCY, "max_tokens": MAX_TOKENS,
    "warmup_concurrency": 1, "tenants": ["team-a", "team-b"], "seed": 7,
    "arrival_spacing_ms": 0, "dtype": DTYPE})


## 5. 渐进压测

四种策略分别扫描短到长 prompt、并发1→16，每点三次，全量成功才列为安全点。目标长度是重复词数，实际 tokenizer 长度从 telemetry 读取；8192 加模板/输出可能先被上下文限制拒绝。429 是过载保护；普通上下文拒绝不是 OOM。观察到 OOM 后停止升压。

报告按 (实际上下文, 并发) 点解释，禁止把不同点的两个最大值拼成安全配置；未见 OOM 只代表本次未触及，不能推断最大显存容量。

In [ ]:
for strategy in STRATEGIES:
    name = "-".join(strategy)
    start_server(strategy, "progressive-" + name)
    try:
        target = OUT / "progressive" / name
        command([PY, "-m", "benchmarks.progressive_load", "--base-url", BASE_URL,
            "--tenant-id", "team-a", "--output-dir", target,
            "--contexts", "32,128,512,2048,4096,8192", "--concurrencies", "1,2,4,8,16",
            "--max-tokens", "16", "--repetitions", "3", "--warmup"],
            "progressive-" + name, timeout=14400)
        report = json.loads((target / "progressive_report.json").read_text())
        print(name, json.dumps(report["safety"], ensure_ascii=False))
    finally:
        stop_server()


## 6. 正式矩阵：生成原始四件套

实际生成 manifest/trace/requests/summary，不改名 progressive 文件。额外保存 observations.json：原始 query/ledger、实际发送时间、完成时间和测量窗口吞吐。trace_id 含规范化 trace 哈希，内容变化即比较失败。

禁止覆盖旧 run；中断后保留半成品，开新会话重跑。warm-up 不参加性能统计。

In [ ]:
formal_runs, measured_runs = [], []
for repetition in range(REPETITIONS):
    offset = repetition % len(STRATEGIES)
    for strategy in STRATEGIES[offset:] + STRATEGIES[:offset]:
        label = "-".join(strategy) + "-" + str(repetition)
        start_server(strategy, label)
        try:
            for warmup in (True, False):
                name = label + ("-warmup" if warmup else "-measured")
                target = OUT / "matrix" / name
                argv = [PY, "-m", "benchmarks.formal_gpu_run", "--root", WORKDIR,
                    "--base-url", BASE_URL, "--output", target, "--run-id", name,
                    "--admission", strategy[0], "--scheduler", strategy[1],
                    "--prefix-mode", strategy[2], "--dtype", DTYPE,
                    "--repetition-index", str(repetition), "--seed", "7",
                    "--tenants", "team-a", "team-b", "--max-tokens", str(MAX_TOKENS),
                    "--concurrency", "1" if warmup else str(CONCURRENCY),
                    "--contexts", *map(str, CONTEXTS)]
                if warmup:
                    argv.append("--warmup")
                command(argv, name)
                formal_runs.append(target)
                if not warmup:
                    measured_runs.append(target)
        finally:
            stop_server()
save("run-index.json", {"formal": [str(p.relative_to(OUT)) for p in formal_runs],
                       "measured": [str(p.relative_to(OUT)) for p in measured_runs]})


## 7. 对照门禁与逐次结果

重新分析原始文件，检查 executor/model/hardware/software/trace 一致性与至少三次测量。MATRIX_VALID 只表示可比较，不意味着性能提高。保留全部重复；估算成本无价格时为 null，不表示免费。原分析器吞吐未计入客户端等待空档，本报告使用 observations 中实际窗口吞吐。

In [ ]:
for path in formal_runs:
    command([PY, "-m", "benchmarks.analyze", "--manifest", path / "manifest.json",
        "--trace", path / "trace.jsonl", "--requests", path / "requests.jsonl",
        "--output", path / "summary.json"], "analyze-" + path.name)
command([PY, "-m", "benchmarks.strategy_matrix", "--metric", "ttft_ms",
         "--output", OUT / "matrix.json", *formal_runs], "matrix-validation")
rows = []
for path in measured_runs:
    summary = json.loads((path / "summary.json").read_text())
    observation = json.loads((path / "observations.json").read_text())
    rows.append({
        "run": path.name, "strategy": path.name.rsplit("-", 2)[0],
        "ttft_p95_ms": summary["metrics"]["ttft_ms"]["p95"],
        "ttft_p99_ms": summary["metrics"]["ttft_ms"]["p99"],
        "tpot_p95_ms": summary["metrics"]["tpot_ms"]["p95"],
        "total_p99_ms": summary["metrics"]["total_ms"]["p99"],
        "wall_tokens_per_s": observation["wall_throughput_tokens_per_s"],
        "fairness_jain": summary["fairness_jain"], "rejection_rate": summary["rejection_rate"],
        "request_kv_peak_blocks": summary["resource_peaks"]["reserved_blocks_peak"],
        "logical_hits": summary["prefix_observation"]["logical_hits"],
        "finished": summary["terminal_counts"].get("FINISHED", 0),
        "failed": summary["terminal_counts"].get("FAILED", 0),
    })
save("comparison-rows.json", rows)
from IPython.display import display, Markdown
import pandas as pd
display(pd.DataFrame(rows))


## 8. 五项故障验证：控制路径与真实 GPU

先运行原有确定性检查，再用独立真实模型进程验证：HTTP取消/重复取消、真实SSE连接关闭、短deadline、model.forward异常、CUDA分配OOM。逐项检查唯一终态、逻辑reservation/active回零、生成线程结束、allocated显存回到基线（容差16MiB）、正常请求恢复。

本单元会**主动触发一次真实 CUDA OOM**：在 model.forward 内申请超过整卡容量的 tensor，必须由 CUDA allocator 抛出 OutOfMemoryError。它验证受控分配失败后的回收，**不是自然负载容量边界或worker进程崩溃**。自然OOM是否出现由渐进扫描回答。异常同样明确标注为受控注入。只影响当前Colab实验进程。

任一项失败都写出 FAIL；若runtime被平台终止，保留Drive已有证据，记为未完成。

In [ ]:
stop_server()
command([PY, "-m", "benchmarks.chaos_validation", "--output", OUT / "chaos-control.json"], "chaos-control")
command([PY, "-m", "benchmarks.colab_phase1", "gpu-chaos",
         "--dtype", DTYPE, "--output", OUT / "chaos-gpu.json"],
        "chaos-gpu", timeout=1800, required=False)
print((OUT / "chaos-gpu.json").read_text() if (OUT / "chaos-gpu.json").exists()
      else "缺少 GPU 故障报告，Phase 1 不得通过")


## 9. Sim 原始运行与 Phase 1 子门禁

真实执行 SimExecutor，保留完整事件与四件套；标记 simulated，不与GPU时延/吞吐混合比较。Sim的GPU秒和成本为null。

现有 phase1_exit.py 仅是子门禁；最终结论还必须结合渐进压测、矩阵、真实故障证据和策略有效性。

In [ ]:
command([PY, "-m", "benchmarks.colab_phase1", "sim", "--reference", measured_runs[0],
         "--output", OUT / "sim"], "sim")
command([PY, "-m", "benchmarks.phase1_exit", "--chaos-report", OUT / "chaos-gpu.json",
         "--output", OUT / "phase1-exit.json", OUT / "sim", *measured_runs],
        "phase1-gate", timeout=1800, required=False)


## 10. 产出四项结论

PASS=原始证据和本流程检查通过；FAIL=验证失败；INCONCLUSIVE=数据不足。prefix-aware 若没有逻辑命中证据，必要对照标为结论不足，不自动勾选出口。未预设性能SLO时不判“性能达标”。容量仅限已完整成功点，逻辑KV为逐请求reservation峰值，非全局显存峰值。

三组策略报告中位数和原始重复，数值差异不直接当因果收益；先核对队列压力/准入原因/命中是否触发了对应策略。

In [ ]:
def read_optional(name):
    path = OUT / name
    return json.loads(path.read_text()) if path.exists() else {}

def succeeded(label):
    return read_optional(label + ".status.json").get("returncode") == 0

safe_points = {}
for strategy in STRATEGIES:
    name = "-".join(strategy)
    report = read_optional("progressive/" + name + "/progressive_report.json")
    safe_points[name] = [point for point in report.get("matrix_points", []) if point["safe"]]
progressive_ok = all(safe_points.values()) and all(succeeded("progressive-" + "-".join(s)) for s in STRATEGIES)
matrix_ok = succeeded("matrix-validation") and len(rows) == 4 * REPETITIONS
prefix_hits = sum(row["logical_hits"] for row in rows if row["strategy"] == "strict-wfq-aware")
matrix_status = ("PASS" if prefix_hits > 0 else "INCONCLUSIVE") if matrix_ok and not sum(row["failed"] for row in rows) else "FAIL"
chaos = read_optional("chaos-gpu.json")
chaos_ok = (succeeded("chaos-control") and succeeded("chaos-gpu") and chaos.get("status") == "PASS"
    and {case["name"] for case in chaos.get("cases", [])} == {"cancel", "disconnect", "timeout", "exception", "oom"})
exit_ok = (progressive_ok and matrix_status == "PASS" and chaos_ok and succeeded("phase0")
           and succeeded("api-regression") and succeeded("phase1-gate"))
conclusion = {
    "session": SESSION, "渐进压测": "PASS" if progressive_ok else "FAIL",
    "必要对照": matrix_status, "故障验证": "PASS" if chaos_ok else "FAIL",
    "Phase1出口": "PASS" if exit_ok else "NOT_PASSED",
    "safe_points_by_strategy": safe_points, "prefix_aware_logical_hits": prefix_hits,
    "limits": [
        "Torch TTFT/TPOT 是整段生成后的 Gateway 交付指标，非逐 token GPU decode 性能",
        "物理 prefix 命中不可观测；无逻辑命中证据时不能声称 prefix 调度收益",
        "OOM 是真实 CUDA allocator 受控注入，自然容量边界见 progressive",
        "成本为估算；未配置 GPU 小时价格时 estimated_cost=null",
        "static vs continuous: N/A；当前 Torch 不支持两种 batching",
        "小样本 P99、同权 tenant token Jain 指数仅描述本固定 workload",
    ]}
save("conclusions.json", conclusion)
text = "# Colab Phase 1 验证结论\n\n"
text += "\n".join("- " + key + ": " + conclusion[key] for key in ("渐进压测", "必要对照", "故障验证", "Phase1出口"))
text += "\n\n## 三组必要对照：正式重复的中位数\n\n"
from statistics import median
pairs = [("Strict vs Adaptive", "strict-fcfs-blind", "adaptive-fcfs-blind"),
         ("FCFS vs WFQ", "strict-fcfs-blind", "strict-wfq-blind"),
         ("prefix-blind vs prefix-aware", "strict-wfq-blind", "strict-wfq-aware")]
for label, left, right in pairs:
    text += "### " + label + "\n\n"
    for field in ("ttft_p95_ms", "tpot_p95_ms", "total_p99_ms", "wall_tokens_per_s",
                  "fairness_jain", "rejection_rate", "request_kv_peak_blocks"):
        groups = [[row[field] for row in rows if row["strategy"] == strategy and row[field] is not None]
                  for strategy in (left, right)]
        values = [median(group) if group else None for group in groups]
        text += "- " + field + ": " + str(values[0]) + " → " + str(values[1]) + "\n"
    text += "\n"
if not prefix_hits:
    text += "prefix-aware 无逻辑命中证据：必要对照结论不足，检查 Gateway → PrefixIndex → Scheduler 接线。\n\n"
text += "## 结论边界\n\n" + "\n".join("- " + limit for limit in conclusion["limits"])
text += "\n\n安全点见 conclusions.json；逐次结果见 comparison-rows.json；原始请求见 matrix/。\n"
(OUT / "report.md").write_text(text)
checkpoint()
display(Markdown(text))


## 11. 归档下载（失败也执行）

保存代码快照、环境/依赖、每个原始run、故障事件、日志、checksums和结论。前面中断则写出 NOT_COMPLETED，绝不沿用旧PASS。下载ZIP后解压到独立runs目录，依据证据更新TODO并链接验收记录。

排查：环境失败检查锁定版本可用性/GPU/空间；启动失败查看server日志/T4 dtype/端口；429是准入保护，不是OOM；分析失败检查原始字段/trace/版本，禁止手改summary过门禁；故障失败查看终态/events/显存/recovery，失败就是本次实验结论。

In [ ]:
if "stop_server" in globals():
    stop_server()
if not (OUT / "conclusions.json").exists():
    save("conclusions.json", {"session": SESSION, "Phase1出口": "NOT_COMPLETED",
                             "reason": "前置步骤中断，检查 status.json 和日志"})
    (OUT / "report.md").write_text("# Phase 1 未完成\n\n前置步骤中断，参见日志；不能勾选 TODO。\n")
for relative in code_files:
    target = OUT / "source-snapshot" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORKDIR / relative, target)
save("checksums.json", {str(path.relative_to(OUT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(OUT.rglob("*")) if path.is_file() and path.name != "checksums.json"})
checkpoint()
archive = shutil.make_archive(str(OUT.parent / (SESSION + "-evidence")), "zip",
                             root_dir=OUT.parent, base_dir=OUT.name)
saved_archive = BACKUP.parent / Path(archive).name
shutil.copy2(archive, saved_archive)
print("证据 ZIP:", saved_archive)
print("结论:", BACKUP / "report.md")
from google.colab import files
files.download(archive)
